# Hyperparameter Tuning — FER (CNN + Transformer models)

Standalone notebook for **short HP runs** (15 epochs). Keeps `model_experiment_CNN_Transformer.ipynb` clean.

| Section | What | Typical time |
|---------|------|--------------|
| **Cell 8** | ViT learning-rate search | ~20 min |
| **Cell 9** | POSTERLite learning-rate search | ~20 min |
| **Cell 10** | Ensemble blend-weight search (val only, no training) | ~1 min |

Same data split (seed 42), augmentation, and Wandb project (`CNN`) as the other notebooks.

**One active toggle per Run All** (Cell 7).

In [ ]:
# Cell 1: Imports, device, Wandb
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

import wandb

KAGGLE_DATA = Path(
    "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/"
)
DATA_DIR = KAGGLE_DATA if KAGGLE_DATA.exists() else Path("data")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | DATA_DIR: {DATA_DIR}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def setup_wandb() -> None:
    api_key = os.environ.get("WANDB_API_KEY")
    if api_key is None:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception:
            pass
    if api_key:
        os.environ["WANDB_API_KEY"] = api_key
        wandb.login(key=api_key, relogin=False)
    else:
        print("WANDB_API_KEY not set — wandb runs may fail; training still works.")


setup_wandb()

In [ ]:
# Cell 2: Dataset
class FERDataset(Dataset):
    IMAGE_SIZE = 48
    NUM_PIXELS = IMAGE_SIZE * IMAGE_SIZE

    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.has_labels = "emotion" in self.dataframe.columns

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        pixel_str = self.dataframe.loc[idx, "pixels"]
        pixels = np.fromstring(pixel_str, sep=" ", dtype=np.float32)
        if pixels.size != self.NUM_PIXELS:
            raise ValueError(f"Expected {self.NUM_PIXELS} pixels, got {pixels.size}")
        image = torch.from_numpy(pixels.reshape(self.IMAGE_SIZE, self.IMAGE_SIZE) / 255.0).unsqueeze(0)
        if self.transform is not None:
            image = self.transform(image)
        if self.has_labels:
            return image, int(self.dataframe.loc[idx, "emotion"])
        return image

In [ ]:
# Cell 3: Splits, augmentation, loaders
BATCH_SIZE = 64
TRAIN_RATIO, VAL_RATIO = 0.70, 0.15

TRAIN_AUGMENT = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
])

df = pd.read_csv(DATA_DIR / "train.csv")
print(f"Loaded {len(df)} samples")

full_dataset = FERDataset(df)
train_size = int(len(full_dataset) * TRAIN_RATIO)
val_size = int(len(full_dataset) * VAL_RATIO)
test_size = len(full_dataset) - train_size - val_size
generator = torch.Generator().manual_seed(RANDOM_SEED)
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)
print(f"Train {len(train_dataset)} | Val {len(val_dataset)} | Test {len(test_dataset)}")


class TransformedSubset(Dataset):
    def __init__(self, subset: Dataset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self) -> int:
        return len(self.subset)

    def __getitem__(self, idx: int):
        image, label = self.subset[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


kwargs = {"num_workers": 0, "pin_memory": torch.cuda.is_available()}
train_loader = DataLoader(TransformedSubset(train_dataset, TRAIN_AUGMENT), batch_size=BATCH_SIZE, shuffle=True, **kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **kwargs)

In [ ]:
# Cell 4: Training utilities (HP runs — val only, no test)
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        bs = labels.size(0)
        running_loss += loss.item() * bs
        correct += (outputs.argmax(1) == labels).sum().item()
        total += bs
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        bs = labels.size(0)
        running_loss += loss.item() * bs
        correct += (outputs.argmax(1) == labels).sum().item()
        total += bs
    return running_loss / total, correct / total


def run_hp_training(
    model, train_loader, val_loader, *, run_name, project, epochs, learning_rate, model_name, extra_config=None,
):
    torch.manual_seed(RANDOM_SEED)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    config = {
        "model": model_name,
        "epochs": epochs,
        "batch_size": train_loader.batch_size,
        "learning_rate": learning_rate,
        "optimizer": "Adam",
        "random_seed": RANDOM_SEED,
        "augmentation": "flip+rotation+affine",
        "hp_search": True,
    }
    if extra_config:
        config.update(extra_config)
    wandb.init(project=project, name=run_name, config=config, reinit=True)
    print(f"HP run {run_name} | lr={learning_rate:g} | {epochs} epochs")
    best_val_acc = 0.0
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        best_val_acc = max(best_val_acc, val_acc)
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })
        print(f"Epoch {epoch:3d}/{epochs} | Train {train_acc*100:.2f}% | Val {val_acc*100:.2f}%")
    wandb.log({"best_val_accuracy": best_val_acc})
    wandb.finish()
    print(f"Finished {run_name} | Best Val: {best_val_acc*100:.2f}%")
    return {"best_val_accuracy": best_val_acc}


WANDB_PROJECT = "CNN"
EPOCHS_HP = 15
DROPOUT = 0.3
CKPT_DIR = Path("/kaggle/working/checkpoints") if Path("/kaggle/working").exists() else Path("checkpoints")


def ckpt_path(arch_name: str) -> Path:
    return CKPT_DIR / f"{arch_name}.pt"

In [ ]:
# Cell 5: Models used in HP search (ViT + POSTERLite)
NUM_CLASSES = 7


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__(nn.Conv2d(in_ch, out_ch, kernel, stride, padding, bias=False),
                         nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        return self.encoder(tokens)


def tokens_from_feature_map(feat: torch.Tensor) -> torch.Tensor:
    return feat.flatten(2).transpose(1, 2)


class CNN_ViT_Hybrid(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBNReLU(1, 32), nn.MaxPool2d(2),
            ConvBNReLU(32, 64), nn.MaxPool2d(2),
            ConvBNReLU(64, d_model), nn.MaxPool2d(2),
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_embed = nn.Parameter(torch.zeros(1, 37, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.transformer = TransformerEncoder(d_model, nhead, num_layers, dropout)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, NUM_CLASSES))

    def forward(self, x):
        feat = self.stem(x)
        tokens = tokens_from_feature_map(feat)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos_embed
        tokens = self.transformer(tokens)
        return self.head(tokens[:, 0])


class POSTERLite(nn.Module):
    def __init__(self, d_model=96, nhead=4, num_layers=2, dropout=0.3, grid=4):
        super().__init__()
        self.grid = grid
        self.branch_s1 = nn.Sequential(ConvBNReLU(1, 32), nn.MaxPool2d(2))
        self.branch_s2 = nn.Sequential(ConvBNReLU(32, 64), nn.MaxPool2d(2))
        self.branch_s3 = nn.Sequential(ConvBNReLU(64, d_model), nn.MaxPool2d(2))
        self.proj1 = nn.Conv2d(32, d_model, 1, bias=False)
        self.proj2 = nn.Conv2d(64, d_model, 1, bias=False)
        self.pool_tokens = nn.AdaptiveAvgPool2d(grid)
        n_tokens = 3 * grid * grid
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.transformer = TransformerEncoder(d_model, nhead, num_layers, dropout)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, NUM_CLASSES))

    def _scale_tokens(self, feat, proj=None):
        if proj is not None:
            feat = proj(feat)
        return tokens_from_feature_map(self.pool_tokens(feat))

    def forward(self, x):
        f1 = self.branch_s1(x)
        f2 = self.branch_s2(f1)
        f3 = self.branch_s3(f2)
        tokens = torch.cat([
            self._scale_tokens(f1, self.proj1),
            self._scale_tokens(f2, self.proj2),
            self._scale_tokens(f3, None),
        ], dim=1)
        tokens = tokens + self.pos_embed
        tokens = self.transformer(tokens)
        return self.head(tokens.mean(dim=1))


HP_MODELS = {"CNN_ViT_Hybrid": CNN_ViT_Hybrid, "POSTERLite": POSTERLite}


def load_ckpt(arch_name: str) -> nn.Module | None:
    path = ckpt_path(arch_name)
    backup = CKPT_DIR.parent / "checkpoints_backup" / f"{arch_name}.pt"
    if not path.exists() and backup.exists():
        path = backup
    if not path.exists():
        print(f"Missing checkpoint: {ckpt_path(arch_name)}")
        return None
    payload = torch.load(path, map_location=device)
    model = HP_MODELS[payload["arch"]]()
    model.load_state_dict(payload["state_dict"])
    print(f"Loaded ← {path}")
    return model.to(device)

In [ ]:
# Cell 6: Toggles — ONE True per Run All
#
# ViT LR search:              RUN_VIT_LR_SEARCH = True
# POSTERLite LR search:       RUN_POSTER_LR_SEARCH = True
# Ensemble blend on val:      RUN_ENSEMBLE_WEIGHT_SEARCH = True  (needs both .pt files)

RUN_VIT_LR_SEARCH = True
RUN_POSTER_LR_SEARCH = False
RUN_ENSEMBLE_WEIGHT_SEARCH = False

In [ ]:
# Cell 7: ViT learning-rate search (15 epochs, val only)
VIT_HP_EPOCHS = EPOCHS_HP
VIT_HP_PREFIX = "ViT_HP"

# lr=1e-3: not retrained — best val through epoch 15 from CNN_ViT_Hybrid_Final (wandb)
RECORDED_VIT_HP = [
    {
        "run_name": f"{VIT_HP_PREFIX}_lr1e-3",
        "learning_rate": 1e-3,
        "epochs": VIT_HP_EPOCHS,
        "best_val_accuracy": 0.5541,
        "source": "CNN_ViT_Hybrid_Final wandb @ epoch 15",
    },
]

VIT_LR_TO_TRAIN = [5e-4, 2e-3]
VIT_LR_NAMES = {5e-4: f"{VIT_HP_PREFIX}_lr5e-4", 2e-3: f"{VIT_HP_PREFIX}_lr2e-3"}

vit_hp_results = list(RECORDED_VIT_HP)

if RUN_VIT_LR_SEARCH:
    for lr in VIT_LR_TO_TRAIN:
        run_name = VIT_LR_NAMES[lr]
        print(f"\n=== ViT | lr={lr:g} | {VIT_HP_EPOCHS} epochs ===")
        metrics = run_hp_training(
            CNN_ViT_Hybrid(), train_loader, val_loader,
            run_name=run_name, project=WANDB_PROJECT, epochs=VIT_HP_EPOCHS,
            learning_rate=lr, model_name="CNN_ViT_Hybrid",
            extra_config={"dropout": DROPOUT, "family": "ViT_LR_search"},
        )
        vit_hp_results.append({
            "run_name": run_name,
            "learning_rate": lr,
            "epochs": VIT_HP_EPOCHS,
            "best_val_accuracy": metrics["best_val_accuracy"],
            "source": "trained",
        })
else:
    print("Skipping ViT LR search.")

if vit_hp_results:
    df_vit_hp = pd.DataFrame(vit_hp_results).sort_values("best_val_accuracy", ascending=False)
    print(f"\n=== ViT LR comparison ({VIT_HP_EPOCHS} ep, best val) ===")
    display(df_vit_hp)
    best = df_vit_hp.iloc[0]
    print(f"Winner: lr={best['learning_rate']:g} | val={best['best_val_accuracy']*100:.2f}%")

In [ ]:
# Cell 8: POSTERLite learning-rate search (15 epochs, val only)
POSTER_HP_EPOCHS = EPOCHS_HP
POSTER_HP_PREFIX = "POSTER_HP"

# lr=1e-3: from POSTERLite_EnsembleCkpt — update after you log epoch-15 val in wandb
RECORDED_POSTER_HP = [
    {
        "run_name": f"{POSTER_HP_PREFIX}_lr1e-3",
        "learning_rate": 1e-3,
        "epochs": POSTER_HP_EPOCHS,
        "best_val_accuracy": 0.5683,
        "source": "POSTERLite_EnsembleCkpt wandb — best val through epoch 15",
    },
]

POSTER_LR_TO_TRAIN = [5e-4, 2e-3]
POSTER_LR_NAMES = {5e-4: f"{POSTER_HP_PREFIX}_lr5e-4", 2e-3: f"{POSTER_HP_PREFIX}_lr2e-3"}

poster_hp_results = list(RECORDED_POSTER_HP)

if RUN_POSTER_LR_SEARCH:
    for lr in POSTER_LR_TO_TRAIN:
        run_name = POSTER_LR_NAMES[lr]
        print(f"\n=== POSTERLite | lr={lr:g} | {POSTER_HP_EPOCHS} epochs ===")
        metrics = run_hp_training(
            POSTERLite(), train_loader, val_loader,
            run_name=run_name, project=WANDB_PROJECT, epochs=POSTER_HP_EPOCHS,
            learning_rate=lr, model_name="POSTERLite",
            extra_config={"dropout": DROPOUT, "family": "POSTER_LR_search"},
        )
        poster_hp_results.append({
            "run_name": run_name,
            "learning_rate": lr,
            "epochs": POSTER_HP_EPOCHS,
            "best_val_accuracy": metrics["best_val_accuracy"],
            "source": "trained",
        })
else:
    print("Skipping POSTERLite LR search.")

if poster_hp_results and RUN_POSTER_LR_SEARCH:
    df_poster_hp = pd.DataFrame(poster_hp_results).sort_values("best_val_accuracy", ascending=False)
    print(f"\n=== POSTERLite LR comparison ({POSTER_HP_EPOCHS} ep, best val) ===")
    display(df_poster_hp)
    best = df_poster_hp.iloc[0]
    print(f"Winner: lr={best['learning_rate']:g} | val={best['best_val_accuracy']*100:.2f}%")

In [ ]:
# Cell 9: Ensemble blend-weight search (val only — no training, ~1 min)
# Needs CNN_ViT_Hybrid.pt + POSTERLite.pt from model_experiment_CNN_Transformer.ipynb
ENSEMBLE_ARCHS = ["CNN_ViT_Hybrid", "POSTERLite"]
VIT_WEIGHTS = [0.3, 0.4, 0.5, 0.6, 0.7]  # POSTER weight = 1 - w_vit

if RUN_ENSEMBLE_WEIGHT_SEARCH:
    models = {}
    for arch in ENSEMBLE_ARCHS:
        m = load_ckpt(arch)
        if m is None:
            raise RuntimeError(
                f"Missing {arch} checkpoint. Train in CNN_Transformer notebook first "
                f"or copy .pt files into {CKPT_DIR}"
            )
        models[arch] = m.eval()

    weight_results = []
    for w_vit in VIT_WEIGHTS:
        w_post = 1.0 - w_vit
        correct, total = 0, 0
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = w_vit * models["CNN_ViT_Hybrid"](images) + w_post * models["POSTERLite"](images)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        val_acc = correct / total
        weight_results.append({"w_vit": w_vit, "w_poster": w_post, "val_accuracy": val_acc})
        print(f"w_vit={w_vit:.1f} w_poster={w_post:.1f} | val={val_acc*100:.2f}%")

    df_w = pd.DataFrame(weight_results).sort_values("val_accuracy", ascending=False)
    print("\n=== Ensemble weight comparison (val) ===")
    display(df_w)
    best = df_w.iloc[0]
    print(f"Best blend: w_vit={best['w_vit']:.1f} | val={best['val_accuracy']*100:.2f}%")

    wandb.init(project=WANDB_PROJECT, name="Ensemble_WeightSearch_Val", reinit=True)
    wandb.log({"weight_results": df_w.to_dict("records"), "best_w_vit": best["w_vit"], "best_val_accuracy": best["val_accuracy"]})
    wandb.finish()
else:
    print("Skipping ensemble weight search.")